In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, mean_absolute_error
)
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_csv('/kaggle/input/resume-dataset/UpdatedResumeDataSet.csv')

# Missing value handling
print("Missing values:\n", df.isnull().sum())
df = df.dropna()  # Drop if any (expected none)

# Text cleaning function
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)  # Remove URLs
    text = re.sub(r'RT|cc', ' ', text)  # Remove RT/cc
    text = re.sub(r'#\S+', '', text)  # Remove hashtags
    text = re.sub(r'@\S+', ' ', text)  # Remove mentions
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)  # Remove punctuation
    text = re.sub(r'[^\x00-\x7f]', ' ', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    return text.lower()

df['cleaned_resume'] = df['Resume'].apply(clean_resume)

# NLP processing: Stopwords removal and lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_process(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['processed_resume'] = df['cleaned_resume'].apply(nlp_process)

# Vectorization
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X = vectorizer.fit_transform(df['processed_resume']).toarray()

# Label encoding for target
le = LabelEncoder()
y = le.fit_transform(df['Category'])
class_labels = le.classes_

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Imbalance handling: Compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Sample correlation matrix (on first 10 features for illustration)
feature_sample = pd.DataFrame(X[:, :10], columns=vectorizer.get_feature_names_out()[:10])
corr_matrix = feature_sample.corr()

# Plot correlation heatmap using matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_matrix.columns)
plt.colorbar(im)
plt.title('Sample Correlation Matrix Heatmap (First 10 Features)')
plt.show()

# Function to evaluate and plot ROC (multi-class OvR)
def evaluate_model(y_test, y_pred, y_prob, model_name, class_labels):
    print(f"\n--- {model_name} ---")
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mae = mean_absolute_error(y_test, y_pred)
    overall_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC: {overall_auc:.4f}")
    print(f"MAE: {mae:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    # Confusion Matrix Plot
    conf = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(conf, cmap='Blues')
    ax.set_xticks(range(len(class_labels)))
    ax.set_yticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=90, ha='right', fontsize=8)
    ax.set_yticklabels(class_labels, fontsize=8)
    plt.colorbar(im)
    # Add annotations (numbers) for clarity, but sparse if too crowded
    for i in range(len(class_labels)):
        for j in range(len(class_labels)):
            if conf[i, j] > 0:  # Only annotate non-zero for less clutter
                ax.text(j, i, conf[i, j], ha='center', va='center', color='black', fontsize=6)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    # ROC Curve (OvR)
    y_bin = label_binarize(y_test, classes=range(len(class_labels)))
    fpr = dict()
    tpr = dict()
    roc_auc_dict = dict()
    for i in range(len(class_labels)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc_dict[i] = auc(fpr[i], tpr[i])
    
    # Plot sample ROC for first class
    plt.figure()
    plt.plot(fpr[0], tpr[0], label=f'Class {class_labels[0]} (AUC = {roc_auc_dict[0]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for {model_name} (Sample Class: {class_labels[0]})')
    plt.legend()
    plt.show()
    # You can add more plots for other classes if needed

# Train LightGBM
lgb_model = lgb.LGBMClassifier(class_weight=class_weight_dict, random_state=42, verbosity=-1)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)
evaluate_model(y_test, y_pred_lgb, y_prob_lgb, "LightGBM", class_labels)

# Train XGBoost (use sample weights)
sample_weights = np.array([class_weight_dict[label] for label in y_train])
xgb_model = xgb.XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)
evaluate_model(y_test, y_pred_xgb, y_prob_xgb, "XGBoost", class_labels)

# Train CatBoost
cat_model = CatBoostClassifier(class_weights=class_weights, random_state=42, verbose=0)
cat_model.fit(X_train, y_train)
y_pred_cat = cat_model.predict(X_test)
y_prob_cat = cat_model.predict_proba(X_test)
evaluate_model(y_test, y_pred_cat, y_prob_cat, "CatBoost", class_labels)

# Ensemble: Stacking without k-fold CV
estimators = [('lgb', lgb_model), ('xgb', xgb_model), ('cat', cat_model)]
stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(class_weight='balanced', random_state=42), cv=None)
stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)
y_prob_stack = stack_model.predict_proba(X_test)
evaluate_model(y_test, y_pred_stack, y_prob_stack, "Ensemble (Stacking)", class_labels)

# Example Tester: Predict on a sample resume using the ensemble model
# Sample resume text (example for a Java Developer)
sample_resume = """
Java Developer with 5 years experience. Skills: Java, Spring, Hibernate, SQL. 
Education: B.Tech in Computer Science. Experience: Developed web applications, managed databases.
"""

# Process the sample resume
cleaned_sample = clean_resume(sample_resume)
processed_sample = nlp_process(cleaned_sample)
X_sample = vectorizer.transform([processed_sample]).toarray()

# Predict
predicted_label = stack_model.predict(X_sample)[0]
predicted_category = le.inverse_transform([predicted_label])[0]
predicted_prob = stack_model.predict_proba(X_sample).max()

print("\n--- Example Tester ---")
print(f"Predicted Category: {predicted_category}")
print(f"Confidence: {predicted_prob:.4f}")

In [ ]:
# Example Tester in a new cell: Predict on another sample resume using the ensemble model
# Sample resume text (example for a Data Scientist)
sample_resume_2 = """
Data Scientist with 3 years experience. Skills: Python, Machine Learning, SQL, Tableau. 
Education: M.Sc in Data Science. Experience: Built predictive models, analyzed large datasets, created visualizations.
"""

# Process the sample resume
cleaned_sample_2 = clean_resume(sample_resume_2)
processed_sample_2 = nlp_process(cleaned_sample_2)
X_sample_2 = vectorizer.transform([processed_sample_2]).toarray()

# Predict
predicted_label_2 = stack_model.predict(X_sample_2)[0]
predicted_category_2 = le.inverse_transform([predicted_label_2])[0]
predicted_prob_2 = stack_model.predict_proba(X_sample_2).max()

print("\n--- Another Example Tester ---")
print(f"Predicted Category: {predicted_category_2}")
print(f"Confidence: {predicted_prob_2:.4f}")

In [ ]:
# 5 More Example Testers for External Validation
# These are synthetic resumes not from the dataset, for validating the model on unseen data.
# Predicting using the ensemble stacking model.

# Sample 1: HR Specialist
sample_resume_1 = """
HR Specialist with 4 years experience. Skills: Recruitment, Employee Relations, Payroll, HRIS. 
Education: B.A. in Human Resources. Experience: Managed hiring processes, resolved employee issues, conducted training sessions.
"""

cleaned_sample_1 = clean_resume(sample_resume_1)
processed_sample_1 = nlp_process(cleaned_sample_1)
X_sample_1 = vectorizer.transform([processed_sample_1]).toarray()
predicted_label_1 = stack_model.predict(X_sample_1)[0]
predicted_category_1 = le.inverse_transform([predicted_label_1])[0]
predicted_prob_1 = stack_model.predict_proba(X_sample_1).max()

print("\n--- External Validation Sample 1 (HR) ---")
print(f"Predicted Category: {predicted_category_1}")
print(f"Confidence: {predicted_prob_1:.4f}")

# Sample 2: Mechanical Engineer
sample_resume_2 = """
Mechanical Engineer with 6 years experience. Skills: CAD, SolidWorks, Thermodynamics, Project Management. 
Education: B.S. in Mechanical Engineering. Experience: Designed mechanical systems, oversaw manufacturing, performed stress analysis.
"""

cleaned_sample_2 = clean_resume(sample_resume_2)
processed_sample_2 = nlp_process(cleaned_sample_2)
X_sample_2 = vectorizer.transform([processed_sample_2]).toarray()
predicted_label_2 = stack_model.predict(X_sample_2)[0]
predicted_category_2 = le.inverse_transform([predicted_label_2])[0]
predicted_prob_2 = stack_model.predict_proba(X_sample_2).max()

print("\n--- External Validation Sample 2 (Mechanical Engineer) ---")
print(f"Predicted Category: {predicted_category_2}")
print(f"Confidence: {predicted_prob_2:.4f}")

# Sample 3: Sales Manager
sample_resume_3 = """
Sales Manager with 5 years experience. Skills: Sales Strategy, CRM, Negotiation, Team Leadership. 
Education: B.B.A. in Marketing. Experience: Led sales teams, achieved revenue targets, built client relationships.
"""

cleaned_sample_3 = clean_resume(sample_resume_3)
processed_sample_3 = nlp_process(cleaned_sample_3)
X_sample_3 = vectorizer.transform([processed_sample_3]).toarray()
predicted_label_3 = stack_model.predict(X_sample_3)[0]
predicted_category_3 = le.inverse_transform([predicted_label_3])[0]
predicted_prob_3 = stack_model.predict_proba(X_sample_3).max()

print("\n--- External Validation Sample 3 (Sales) ---")
print(f"Predicted Category: {predicted_category_3}")
print(f"Confidence: {predicted_prob_3:.4f}")

# Sample 4: Web Designer
sample_resume_4 = """
Web Designer with 3 years experience. Skills: HTML, CSS, JavaScript, Adobe XD, UI/UX Design. 
Education: B.F.A. in Graphic Design. Experience: Created responsive websites, designed user interfaces, optimized for SEO.
"""

cleaned_sample_4 = clean_resume(sample_resume_4)
processed_sample_4 = nlp_process(cleaned_sample_4)
X_sample_4 = vectorizer.transform([processed_sample_4]).toarray()
predicted_label_4 = stack_model.predict(X_sample_4)[0]
predicted_category_4 = le.inverse_transform([predicted_label_4])[0]
predicted_prob_4 = stack_model.predict_proba(X_sample_4).max()

print("\n--- External Validation Sample 4 (Web Designing) ---")
print(f"Predicted Category: {predicted_category_4}")
print(f"Confidence: {predicted_prob_4:.4f}")

# Sample 5: DevOps Engineer
sample_resume_5 = """
DevOps Engineer with 4 years experience. Skills: AWS, Docker, Kubernetes, CI/CD, Jenkins. 
Education: B.Tech in Information Technology. Experience: Automated deployments, managed cloud infrastructure, implemented monitoring tools.
"""

cleaned_sample_5 = clean_resume(sample_resume_5)
processed_sample_5 = nlp_process(cleaned_sample_5)
X_sample_5 = vectorizer.transform([processed_sample_5]).toarray()
predicted_label_5 = stack_model.predict(X_sample_5)[0]
predicted_category_5 = le.inverse_transform([predicted_label_5])[0]
predicted_prob_5 = stack_model.predict_proba(X_sample_5).max()

print("\n--- External Validation Sample 5 (DevOps Engineer) ---")
print(f"Predicted Category: {predicted_category_5}")
print(f"Confidence: {predicted_prob_5:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, mean_absolute_error
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_csv('/kaggle/input/resume-dataset/UpdatedResumeDataSet.csv')

# Missing value handling
print("Missing values:\n", df.isnull().sum())
df = df.dropna()  # Drop if any (expected none)

# Text cleaning function
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)  # Remove URLs
    text = re.sub(r'RT|cc', ' ', text)  # Remove RT/cc
    text = re.sub(r'#\S+', '', text)  # Remove hashtags
    text = re.sub(r'@\S+', ' ', text)  # Remove mentions
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)  # Remove punctuation
    text = re.sub(r'[^\x00-\x7f]', ' ', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    return text.lower()

df['cleaned_resume'] = df['Resume'].apply(clean_resume)

# NLP processing: Stopwords removal and lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_process(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['processed_resume'] = df['cleaned_resume'].apply(nlp_process)

# Label encoding for target
le = LabelEncoder()
y = le.fit_transform(df['Category'])
class_labels = le.classes_

# Tokenization for LSTM
max_words = 5000  # Vocabulary size
max_len = 500  # Max sequence length
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['processed_resume'])
X_sequences = tokenizer.texts_to_sequences(df['processed_resume'])
X_padded = pad_sequences(X_sequences, maxlen=max_len)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42, stratify=y)

# One-hot encode labels for multi-class
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

# Imbalance handling: Compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Build LSTM Model
embedding_dim = 128
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dropout(0.5))
model.add(Dense(len(class_labels), activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train LSTM
history = model.fit(X_train, y_train_cat, epochs=10, batch_size=32, validation_split=0.1, class_weight=class_weight_dict)

# Predict on test set
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# Function to evaluate and plot
def evaluate_model(y_test, y_pred, y_prob, model_name, class_labels):
    print(f"\n--- {model_name} ---")
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mae = mean_absolute_error(y_test, y_pred)
    overall_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC: {overall_auc:.4f}")
    print(f"MAE: {mae:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    # Confusion Matrix Plot
    conf = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(conf, cmap='Blues')
    ax.set_xticks(range(len(class_labels)))
    ax.set_yticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=90, ha='right', fontsize=8)
    ax.set_yticklabels(class_labels, fontsize=8)
    plt.colorbar(im)
    for i in range(len(class_labels)):
        for j in range(len(class_labels)):
            if conf[i, j] > 0:
                ax.text(j, i, conf[i, j], ha='center', va='center', color='black', fontsize=6)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    # ROC Curve (OvR) sample
    y_bin = label_binarize(y_test, classes=range(len(class_labels)))
    fpr = dict()
    tpr = dict()
    roc_auc_dict = dict()
    for i in range(len(class_labels)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc_dict[i] = auc(fpr[i], tpr[i])
    
    plt.figure()
    plt.plot(fpr[0], tpr[0], label=f'Class {class_labels[0]} (AUC = {roc_auc_dict[0]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for {model_name} (Sample Class: {class_labels[0]})')
    plt.legend()
    plt.show()

# Evaluate
evaluate_model(y_test, y_pred, y_pred_prob, "LSTM", class_labels)

# Example Tester: Predict on a sample resume using the LSTM model
sample_resume = """
Java Developer with 5 years experience. Skills: Java, Spring, Hibernate, SQL. 
Education: B.Tech in Computer Science. Experience: Developed web applications, managed databases.
"""

# Process the sample resume
cleaned_sample = clean_resume(sample_resume)
processed_sample = nlp_process(cleaned_sample)
seq_sample = tokenizer.texts_to_sequences([processed_sample])
pad_sample = pad_sequences(seq_sample, maxlen=max_len)
pred_prob = model.predict(pad_sample)
predicted_label = np.argmax(pred_prob, axis=1)[0]
predicted_category = le.inverse_transform([predicted_label])[0]
confidence = pred_prob.max()

print("\n--- Example Tester ---")
print(f"Predicted Category: {predicted_category}")
print(f"Confidence: {confidence:.4f}")

In [ ]:
# 5 More Example Testers for External Validation
# These are synthetic resumes not from the dataset, for validating the model on unseen data.
# Predicting using the ensemble stacking model.

# Sample 1: HR Specialist
sample_resume_1 = """
HR Specialist with 4 years experience. Skills: Recruitment, Employee Relations, Payroll, HRIS. 
Education: B.A. in Human Resources. Experience: Managed hiring processes, resolved employee issues, conducted training sessions.
"""

cleaned_sample_1 = clean_resume(sample_resume_1)
processed_sample_1 = nlp_process(cleaned_sample_1)
X_sample_1 = vectorizer.transform([processed_sample_1]).toarray()
predicted_label_1 = stack_model.predict(X_sample_1)[0]
predicted_category_1 = le.inverse_transform([predicted_label_1])[0]
predicted_prob_1 = stack_model.predict_proba(X_sample_1).max()

print("\n--- External Validation Sample 1 (HR) ---")
print(f"Predicted Category: {predicted_category_1}")
print(f"Confidence: {predicted_prob_1:.4f}")

# Sample 2: Mechanical Engineer
sample_resume_2 = """
Mechanical Engineer with 6 years experience. Skills: CAD, SolidWorks, Thermodynamics, Project Management. 
Education: B.S. in Mechanical Engineering. Experience: Designed mechanical systems, oversaw manufacturing, performed stress analysis.
"""

cleaned_sample_2 = clean_resume(sample_resume_2)
processed_sample_2 = nlp_process(cleaned_sample_2)
X_sample_2 = vectorizer.transform([processed_sample_2]).toarray()
predicted_label_2 = stack_model.predict(X_sample_2)[0]
predicted_category_2 = le.inverse_transform([predicted_label_2])[0]
predicted_prob_2 = stack_model.predict_proba(X_sample_2).max()

print("\n--- External Validation Sample 2 (Mechanical Engineer) ---")
print(f"Predicted Category: {predicted_category_2}")
print(f"Confidence: {predicted_prob_2:.4f}")

# Sample 3: Sales Manager
sample_resume_3 = """
Sales Manager with 5 years experience. Skills: Sales Strategy, CRM, Negotiation, Team Leadership. 
Education: B.B.A. in Marketing. Experience: Led sales teams, achieved revenue targets, built client relationships.
"""

cleaned_sample_3 = clean_resume(sample_resume_3)
processed_sample_3 = nlp_process(cleaned_sample_3)
X_sample_3 = vectorizer.transform([processed_sample_3]).toarray()
predicted_label_3 = stack_model.predict(X_sample_3)[0]
predicted_category_3 = le.inverse_transform([predicted_label_3])[0]
predicted_prob_3 = stack_model.predict_proba(X_sample_3).max()

print("\n--- External Validation Sample 3 (Sales) ---")
print(f"Predicted Category: {predicted_category_3}")
print(f"Confidence: {predicted_prob_3:.4f}")

# Sample 4: Web Designer
sample_resume_4 = """
Web Designer with 3 years experience. Skills: HTML, CSS, JavaScript, Adobe XD, UI/UX Design. 
Education: B.F.A. in Graphic Design. Experience: Created responsive websites, designed user interfaces, optimized for SEO.
"""

cleaned_sample_4 = clean_resume(sample_resume_4)
processed_sample_4 = nlp_process(cleaned_sample_4)
X_sample_4 = vectorizer.transform([processed_sample_4]).toarray()
predicted_label_4 = stack_model.predict(X_sample_4)[0]
predicted_category_4 = le.inverse_transform([predicted_label_4])[0]
predicted_prob_4 = stack_model.predict_proba(X_sample_4).max()

print("\n--- External Validation Sample 4 (Web Designing) ---")
print(f"Predicted Category: {predicted_category_4}")
print(f"Confidence: {predicted_prob_4:.4f}")

# Sample 5: DevOps Engineer
sample_resume_5 = """
DevOps Engineer with 4 years experience. Skills: AWS, Docker, Kubernetes, CI/CD, Jenkins. 
Education: B.Tech in Information Technology. Experience: Automated deployments, managed cloud infrastructure, implemented monitoring tools.
"""

cleaned_sample_5 = clean_resume(sample_resume_5)
processed_sample_5 = nlp_process(cleaned_sample_5)
X_sample_5 = vectorizer.transform([processed_sample_5]).toarray()
predicted_label_5 = stack_model.predict(X_sample_5)[0]
predicted_category_5 = le.inverse_transform([predicted_label_5])[0]
predicted_prob_5 = stack_model.predict_proba(X_sample_5).max()

print("\n--- External Validation Sample 5 (DevOps Engineer) ---")
print(f"Predicted Category: {predicted_category_5}")
print(f"Confidence: {predicted_prob_5:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, mean_absolute_error
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_csv('/kaggle/input/resume-dataset/UpdatedResumeDataSet.csv')

# Missing value handling
print("Missing values:\n", df.isnull().sum())
df = df.dropna()  # Drop if any (expected none)

# Text cleaning function
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)  # Remove URLs
    text = re.sub(r'RT|cc', ' ', text)  # Remove RT/cc
    text = re.sub(r'#\S+', '', text)  # Remove hashtags
    text = re.sub(r'@\S+', ' ', text)  # Remove mentions
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)  # Remove punctuation
    text = re.sub(r'[^\x00-\x7f]', ' ', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    return text.lower()

df['cleaned_resume'] = df['Resume'].apply(clean_resume)

# NLP processing: Stopwords removal and lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_process(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['processed_resume'] = df['cleaned_resume'].apply(nlp_process)

# Vectorization with TF-IDF (suitable for MLP input)
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X = vectorizer.fit_transform(df['processed_resume']).toarray()

# Label encoding for target
le = LabelEncoder()
y = le.fit_transform(df['Category'])
class_labels = le.classes_

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# One-hot encode labels for multi-class
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

# Imbalance handling: Compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Build MLP Model (Multi-Layer Perceptron using Keras)
model = Sequential()
model.add(Dense(512, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(class_labels), activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train MLP
history = model.fit(X_train, y_train_cat, epochs=20, batch_size=32, validation_split=0.1, class_weight=class_weight_dict)

# Predict on test set
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# Function to evaluate and plot
def evaluate_model(y_test, y_pred, y_prob, model_name, class_labels):
    print(f"\n--- {model_name} ---")
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mae = mean_absolute_error(y_test, y_pred)
    overall_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC: {overall_auc:.4f}")
    print(f"MAE: {mae:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    # Confusion Matrix Plot
    conf = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(conf, cmap='Blues')
    ax.set_xticks(range(len(class_labels)))
    ax.set_yticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=90, ha='right', fontsize=8)
    ax.set_yticklabels(class_labels, fontsize=8)
    plt.colorbar(im)
    for i in range(len(class_labels)):
        for j in range(len(class_labels)):
            if conf[i, j] > 0:
                ax.text(j, i, conf[i, j], ha='center', va='center', color='black', fontsize=6)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    # ROC Curve (OvR) sample
    y_bin = label_binarize(y_test, classes=range(len(class_labels)))
    fpr = dict()
    tpr = dict()
    roc_auc_dict = dict()
    for i in range(len(class_labels)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc_dict[i] = auc(fpr[i], tpr[i])
    
    plt.figure()
    plt.plot(fpr[0], tpr[0], label=f'Class {class_labels[0]} (AUC = {roc_auc_dict[0]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for {model_name} (Sample Class: {class_labels[0]})')
    plt.legend()
    plt.show()

# Evaluate
evaluate_model(y_test, y_pred, y_pred_prob, "MLP", class_labels)

# Example Tester: Predict on a sample resume using the MLP model
sample_resume = """
Java Developer with 5 years experience. Skills: Java, Spring, Hibernate, SQL. 
Education: B.Tech in Computer Science. Experience: Developed web applications, managed databases.
"""

# Process the sample resume
cleaned_sample = clean_resume(sample_resume)
processed_sample = nlp_process(cleaned_sample)
X_sample = vectorizer.transform([processed_sample]).toarray()
pred_prob = model.predict(X_sample)
predicted_label = np.argmax(pred_prob, axis=1)[0]
predicted_category = le.inverse_transform([predicted_label])[0]
confidence = pred_prob.max()

print("\n--- Example Tester ---")
print(f"Predicted Category: {predicted_category}")
print(f"Confidence: {confidence:.4f}")

In [ ]:
# 5 More Example Testers for External Validation
# These are synthetic resumes not from the dataset, for validating the model on unseen data.
# Predicting using the ensemble stacking model.

# Sample 1: HR Specialist
sample_resume_1 = """
HR Specialist with 4 years experience. Skills: Recruitment, Employee Relations, Payroll, HRIS. 
Education: B.A. in Human Resources. Experience: Managed hiring processes, resolved employee issues, conducted training sessions.
"""

cleaned_sample_1 = clean_resume(sample_resume_1)
processed_sample_1 = nlp_process(cleaned_sample_1)
X_sample_1 = vectorizer.transform([processed_sample_1]).toarray()
predicted_label_1 = stack_model.predict(X_sample_1)[0]
predicted_category_1 = le.inverse_transform([predicted_label_1])[0]
predicted_prob_1 = stack_model.predict_proba(X_sample_1).max()

print("\n--- External Validation Sample 1 (HR) ---")
print(f"Predicted Category: {predicted_category_1}")
print(f"Confidence: {predicted_prob_1:.4f}")

# Sample 2: Mechanical Engineer
sample_resume_2 = """
Mechanical Engineer with 6 years experience. Skills: CAD, SolidWorks, Thermodynamics, Project Management. 
Education: B.S. in Mechanical Engineering. Experience: Designed mechanical systems, oversaw manufacturing, performed stress analysis.
"""

cleaned_sample_2 = clean_resume(sample_resume_2)
processed_sample_2 = nlp_process(cleaned_sample_2)
X_sample_2 = vectorizer.transform([processed_sample_2]).toarray()
predicted_label_2 = stack_model.predict(X_sample_2)[0]
predicted_category_2 = le.inverse_transform([predicted_label_2])[0]
predicted_prob_2 = stack_model.predict_proba(X_sample_2).max()

print("\n--- External Validation Sample 2 (Mechanical Engineer) ---")
print(f"Predicted Category: {predicted_category_2}")
print(f"Confidence: {predicted_prob_2:.4f}")

# Sample 3: Sales Manager
sample_resume_3 = """
Sales Manager with 5 years experience. Skills: Sales Strategy, CRM, Negotiation, Team Leadership. 
Education: B.B.A. in Marketing. Experience: Led sales teams, achieved revenue targets, built client relationships.
"""

cleaned_sample_3 = clean_resume(sample_resume_3)
processed_sample_3 = nlp_process(cleaned_sample_3)
X_sample_3 = vectorizer.transform([processed_sample_3]).toarray()
predicted_label_3 = stack_model.predict(X_sample_3)[0]
predicted_category_3 = le.inverse_transform([predicted_label_3])[0]
predicted_prob_3 = stack_model.predict_proba(X_sample_3).max()

print("\n--- External Validation Sample 3 (Sales) ---")
print(f"Predicted Category: {predicted_category_3}")
print(f"Confidence: {predicted_prob_3:.4f}")

# Sample 4: Web Designer
sample_resume_4 = """
Web Designer with 3 years experience. Skills: HTML, CSS, JavaScript, Adobe XD, UI/UX Design. 
Education: B.F.A. in Graphic Design. Experience: Created responsive websites, designed user interfaces, optimized for SEO.
"""

cleaned_sample_4 = clean_resume(sample_resume_4)
processed_sample_4 = nlp_process(cleaned_sample_4)
X_sample_4 = vectorizer.transform([processed_sample_4]).toarray()
predicted_label_4 = stack_model.predict(X_sample_4)[0]
predicted_category_4 = le.inverse_transform([predicted_label_4])[0]
predicted_prob_4 = stack_model.predict_proba(X_sample_4).max()

print("\n--- External Validation Sample 4 (Web Designing) ---")
print(f"Predicted Category: {predicted_category_4}")
print(f"Confidence: {predicted_prob_4:.4f}")

# Sample 5: DevOps Engineer
sample_resume_5 = """
DevOps Engineer with 4 years experience. Skills: AWS, Docker, Kubernetes, CI/CD, Jenkins. 
Education: B.Tech in Information Technology. Experience: Automated deployments, managed cloud infrastructure, implemented monitoring tools.
"""

cleaned_sample_5 = clean_resume(sample_resume_5)
processed_sample_5 = nlp_process(cleaned_sample_5)
X_sample_5 = vectorizer.transform([processed_sample_5]).toarray()
predicted_label_5 = stack_model.predict(X_sample_5)[0]
predicted_category_5 = le.inverse_transform([predicted_label_5])[0]
predicted_prob_5 = stack_model.predict_proba(X_sample_5).max()

print("\n--- External Validation Sample 5 (DevOps Engineer) ---")
print(f"Predicted Category: {predicted_category_5}")
print(f"Confidence: {predicted_prob_5:.4f}")

In [ ]:
# 20 Examples for External Validation
# These are synthetic resumes not from the dataset, for validating the model on unseen data.
# Predicting using the MLP model.

# Sample 1: Advocate
sample_resume_1 = """
Advocate with 7 years experience. Skills: Legal Research, Court Advocacy, Contract Drafting, Negotiation. 
Education: LLB in Law. Experience: Represented clients in court, drafted legal documents, advised on compliance.
"""
cleaned_sample_1 = clean_resume(sample_resume_1)
processed_sample_1 = nlp_process(cleaned_sample_1)
X_sample_1 = vectorizer.transform([processed_sample_1]).toarray()
pred_prob_1 = model.predict(X_sample_1)
predicted_label_1 = np.argmax(pred_prob_1, axis=1)[0]
predicted_category_1 = le.inverse_transform([predicted_label_1])[0]
confidence_1 = pred_prob_1.max()
print("\n--- External Validation Sample 1 (Advocate) ---")
print(f"Predicted Category: {predicted_category_1}")
print(f"Confidence: {confidence_1:.4f}")

# Sample 2: Arts
sample_resume_2 = """
Artist with 5 years experience. Skills: Painting, Sculpture, Digital Art, Adobe Creative Suite. 
Education: BFA in Fine Arts. Experience: Exhibited works in galleries, created custom art pieces, taught art classes.
"""
cleaned_sample_2 = clean_resume(sample_resume_2)
processed_sample_2 = nlp_process(cleaned_sample_2)
X_sample_2 = vectorizer.transform([processed_sample_2]).toarray()
pred_prob_2 = model.predict(X_sample_2)
predicted_label_2 = np.argmax(pred_prob_2, axis=1)[0]
predicted_category_2 = le.inverse_transform([predicted_label_2])[0]
confidence_2 = pred_prob_2.max()
print("\n--- External Validation Sample 2 (Arts) ---")
print(f"Predicted Category: {predicted_category_2}")
print(f"Confidence: {confidence_2:.4f}")

# Sample 3: Automation Testing
sample_resume_3 = """
Automation Tester with 4 years experience. Skills: Selenium, Appium, JUnit, TestNG. 
Education: B.Tech in Computer Science. Experience: Developed automated test scripts, executed regression tests, reported bugs.
"""
cleaned_sample_3 = clean_resume(sample_resume_3)
processed_sample_3 = nlp_process(cleaned_sample_3)
X_sample_3 = vectorizer.transform([processed_sample_3]).toarray()
pred_prob_3 = model.predict(X_sample_3)
predicted_label_3 = np.argmax(pred_prob_3, axis=1)[0]
predicted_category_3 = le.inverse_transform([predicted_label_3])[0]
confidence_3 = pred_prob_3.max()
print("\n--- External Validation Sample 3 (Automation Testing) ---")
print(f"Predicted Category: {predicted_category_3}")
print(f"Confidence: {confidence_3:.4f}")

# Sample 4: Blockchain
sample_resume_4 = """
Blockchain Developer with 3 years experience. Skills: Solidity, Ethereum, Hyperledger, Smart Contracts. 
Education: B.S. in Computer Engineering. Experience: Built decentralized apps, implemented blockchain solutions, audited smart contracts.
"""
cleaned_sample_4 = clean_resume(sample_resume_4)
processed_sample_4 = nlp_process(cleaned_sample_4)
X_sample_4 = vectorizer.transform([processed_sample_4]).toarray()
pred_prob_4 = model.predict(X_sample_4)
predicted_label_4 = np.argmax(pred_prob_4, axis=1)[0]
predicted_category_4 = le.inverse_transform([predicted_label_4])[0]
confidence_4 = pred_prob_4.max()
print("\n--- External Validation Sample 4 (Blockchain) ---")
print(f"Predicted Category: {predicted_category_4}")
print(f"Confidence: {confidence_4:.4f}")

# Sample 5: Business Analyst
sample_resume_5 = """
Business Analyst with 6 years experience. Skills: Requirements Gathering, Data Analysis, UML, Agile. 
Education: MBA in Business Administration. Experience: Analyzed business processes, created use cases, facilitated stakeholder meetings.
"""
cleaned_sample_5 = clean_resume(sample_resume_5)
processed_sample_5 = nlp_process(cleaned_sample_5)
X_sample_5 = vectorizer.transform([processed_sample_5]).toarray()
pred_prob_5 = model.predict(X_sample_5)
predicted_label_5 = np.argmax(pred_prob_5, axis=1)[0]
predicted_category_5 = le.inverse_transform([predicted_label_5])[0]
confidence_5 = pred_prob_5.max()
print("\n--- External Validation Sample 5 (Business Analyst) ---")
print(f"Predicted Category: {predicted_category_5}")
print(f"Confidence: {confidence_5:.4f}")

# Sample 6: Civil Engineer
sample_resume_6 = """
Civil Engineer with 5 years experience. Skills: AutoCAD, Structural Analysis, Project Management, Surveying. 
Education: B.E. in Civil Engineering. Experience: Designed infrastructure projects, supervised construction sites, ensured safety compliance.
"""
cleaned_sample_6 = clean_resume(sample_resume_6)
processed_sample_6 = nlp_process(cleaned_sample_6)
X_sample_6 = vectorizer.transform([processed_sample_6]).toarray()
pred_prob_6 = model.predict(X_sample_6)
predicted_label_6 = np.argmax(pred_prob_6, axis=1)[0]
predicted_category_6 = le.inverse_transform([predicted_label_6])[0]
confidence_6 = pred_prob_6.max()
print("\n--- External Validation Sample 6 (Civil Engineer) ---")
print(f"Predicted Category: {predicted_category_6}")
print(f"Confidence: {confidence_6:.4f}")

# Sample 7: Data Science
sample_resume_7 = """
Data Scientist with 4 years experience. Skills: Python, R, Machine Learning, SQL, Tableau. 
Education: M.S. in Data Science. Experience: Built predictive models, analyzed big data, created dashboards.
"""
cleaned_sample_7 = clean_resume(sample_resume_7)
processed_sample_7 = nlp_process(cleaned_sample_7)
X_sample_7 = vectorizer.transform([processed_sample_7]).toarray()
pred_prob_7 = model.predict(X_sample_7)
predicted_label_7 = np.argmax(pred_prob_7, axis=1)[0]
predicted_category_7 = le.inverse_transform([predicted_label_7])[0]
confidence_7 = pred_prob_7.max()
print("\n--- External Validation Sample 7 (Data Science) ---")
print(f"Predicted Category: {predicted_category_7}")
print(f"Confidence: {confidence_7:.4f}")

# Sample 8: Database
sample_resume_8 = """
Database Administrator with 5 years experience. Skills: SQL Server, Oracle, MySQL, Database Optimization. 
Education: B.Tech in Information Technology. Experience: Managed databases, performed backups, optimized queries.
"""
cleaned_sample_8 = clean_resume(sample_resume_8)
processed_sample_8 = nlp_process(cleaned_sample_8)
X_sample_8 = vectorizer.transform([processed_sample_8]).toarray()
pred_prob_8 = model.predict(X_sample_8)
predicted_label_8 = np.argmax(pred_prob_8, axis=1)[0]
predicted_category_8 = le.inverse_transform([predicted_label_8])[0]
confidence_8 = pred_prob_8.max()
print("\n--- External Validation Sample 8 (Database) ---")
print(f"Predicted Category: {predicted_category_8}")
print(f"Confidence: {confidence_8:.4f}")

# Sample 9: DevOps Engineer
sample_resume_9 = """
DevOps Engineer with 4 years experience. Skills: AWS, Docker, Kubernetes, Jenkins, Terraform. 
Education: B.S. in Computer Science. Experience: Automated CI/CD pipelines, managed cloud infrastructure, monitored systems.
"""
cleaned_sample_9 = clean_resume(sample_resume_9)
processed_sample_9 = nlp_process(cleaned_sample_9)
X_sample_9 = vectorizer.transform([processed_sample_9]).toarray()
pred_prob_9 = model.predict(X_sample_9)
predicted_label_9 = np.argmax(pred_prob_9, axis=1)[0]
predicted_category_9 = le.inverse_transform([predicted_label_9])[0]
confidence_9 = pred_prob_9.max()
print("\n--- External Validation Sample 9 (DevOps Engineer) ---")
print(f"Predicted Category: {predicted_category_9}")
print(f"Confidence: {confidence_9:.4f}")

# Sample 10: DotNet Developer
sample_resume_10 = """
DotNet Developer with 6 years experience. Skills: C#, ASP.NET, MVC, SQL Server. 
Education: B.Tech in Software Engineering. Experience: Developed web applications, integrated APIs, maintained codebases.
"""
cleaned_sample_10 = clean_resume(sample_resume_10)
processed_sample_10 = nlp_process(cleaned_sample_10)
X_sample_10 = vectorizer.transform([processed_sample_10]).toarray()
pred_prob_10 = model.predict(X_sample_10)
predicted_label_10 = np.argmax(pred_prob_10, axis=1)[0]
predicted_category_10 = le.inverse_transform([predicted_label_10])[0]
confidence_10 = pred_prob_10.max()
print("\n--- External Validation Sample 10 (DotNet Developer) ---")
print(f"Predicted Category: {predicted_category_10}")
print(f"Confidence: {confidence_10:.4f}")

# Sample 11: ETL Developer
sample_resume_11 = """
ETL Developer with 5 years experience. Skills: Informatica, Talend, SQL, Data Warehousing. 
Education: B.E. in Computer Engineering. Experience: Designed ETL processes, extracted data from sources, transformed and loaded into warehouses.
"""
cleaned_sample_11 = clean_resume(sample_resume_11)
processed_sample_11 = nlp_process(cleaned_sample_11)
X_sample_11 = vectorizer.transform([processed_sample_11]).toarray()
pred_prob_11 = model.predict(X_sample_11)
predicted_label_11 = np.argmax(pred_prob_11, axis=1)[0]
predicted_category_11 = le.inverse_transform([predicted_label_11])[0]
confidence_11 = pred_prob_11.max()
print("\n--- External Validation Sample 11 (ETL Developer) ---")
print(f"Predicted Category: {predicted_category_11}")
print(f"Confidence: {confidence_11:.4f}")

# Sample 12: Electrical Engineering
sample_resume_12 = """
Electrical Engineer with 4 years experience. Skills: Circuit Design, PCB Layout, MATLAB, Power Systems. 
Education: B.S. in Electrical Engineering. Experience: Designed electrical systems, tested prototypes, troubleshot issues.
"""
cleaned_sample_12 = clean_resume(sample_resume_12)
processed_sample_12 = nlp_process(cleaned_sample_12)
X_sample_12 = vectorizer.transform([processed_sample_12]).toarray()
pred_prob_12 = model.predict(X_sample_12)
predicted_label_12 = np.argmax(pred_prob_12, axis=1)[0]
predicted_category_12 = le.inverse_transform([predicted_label_12])[0]
confidence_12 = pred_prob_12.max()
print("\n--- External Validation Sample 12 (Electrical Engineering) ---")
print(f"Predicted Category: {predicted_category_12}")
print(f"Confidence: {confidence_12:.4f}")

# Sample 13: HR
sample_resume_13 = """
HR Manager with 6 years experience. Skills: Talent Acquisition, Performance Management, Employee Engagement, HR Policies. 
Education: MBA in Human Resources. Experience: Recruited staff, conducted appraisals, organized team-building events.
"""
cleaned_sample_13 = clean_resume(sample_resume_13)
processed_sample_13 = nlp_process(cleaned_sample_13)
X_sample_13 = vectorizer.transform([processed_sample_13]).toarray()
pred_prob_13 = model.predict(X_sample_13)
predicted_label_13 = np.argmax(pred_prob_13, axis=1)[0]
predicted_category_13 = le.inverse_transform([predicted_label_13])[0]
confidence_13 = pred_prob_13.max()
print("\n--- External Validation Sample 13 (HR) ---")
print(f"Predicted Category: {predicted_category_13}")
print(f"Confidence: {confidence_13:.4f}")

# Sample 14: Hadoop
sample_resume_14 = """
Hadoop Developer with 5 years experience. Skills: HDFS, MapReduce, Hive, Pig. 
Education: B.Tech in Data Engineering. Experience: Managed big data clusters, processed large datasets, optimized Hadoop jobs.
"""
cleaned_sample_14 = clean_resume(sample_resume_14)
processed_sample_14 = nlp_process(cleaned_sample_14)
X_sample_14 = vectorizer.transform([processed_sample_14]).toarray()
pred_prob_14 = model.predict(X_sample_14)
predicted_label_14 = np.argmax(pred_prob_14, axis=1)[0]
predicted_category_14 = le.inverse_transform([predicted_label_14])[0]
confidence_14 = pred_prob_14.max()
print("\n--- External Validation Sample 14 (Hadoop) ---")
print(f"Predicted Category: {predicted_category_14}")
print(f"Confidence: {confidence_14:.4f}")

# Sample 15: Health and fitness
sample_resume_15 = """
Fitness Trainer with 4 years experience. Skills: Personal Training, Nutrition Planning, Group Fitness, CPR Certified. 
Education: Certification in Fitness Training. Experience: Trained clients, designed workout programs, promoted healthy lifestyles.
"""
cleaned_sample_15 = clean_resume(sample_resume_15)
processed_sample_15 = nlp_process(cleaned_sample_15)
X_sample_15 = vectorizer.transform([processed_sample_15]).toarray()
pred_prob_15 = model.predict(X_sample_15)
predicted_label_15 = np.argmax(pred_prob_15, axis=1)[0]
predicted_category_15 = le.inverse_transform([predicted_label_15])[0]
confidence_15 = pred_prob_15.max()
print("\n--- External Validation Sample 15 (Health and fitness) ---")
print(f"Predicted Category: {predicted_category_15}")
print(f"Confidence: {confidence_15:.4f}")

# Sample 16: Java Developer
sample_resume_16 = """
Java Developer with 5 years experience. Skills: Java, Spring Boot, Microservices, REST APIs. 
Education: B.S. in Computer Science. Experience: Built enterprise applications, integrated databases, deployed to cloud.
"""
cleaned_sample_16 = clean_resume(sample_resume_16)
processed_sample_16 = nlp_process(cleaned_sample_16)
X_sample_16 = vectorizer.transform([processed_sample_16]).toarray()
pred_prob_16 = model.predict(X_sample_16)
predicted_label_16 = np.argmax(pred_prob_16, axis=1)[0]
predicted_category_16 = le.inverse_transform([predicted_label_16])[0]
confidence_16 = pred_prob_16.max()
print("\n--- External Validation Sample 16 (Java Developer) ---")
print(f"Predicted Category: {predicted_category_16}")
print(f"Confidence: {confidence_16:.4f}")

# Sample 17: Mechanical Engineer
sample_resume_17 = """
Mechanical Engineer with 6 years experience. Skills: SolidWorks, ANSYS, Thermodynamics, Manufacturing Processes. 
Education: B.E. in Mechanical Engineering. Experience: Designed mechanical components, conducted simulations, managed production.
"""
cleaned_sample_17 = clean_resume(sample_resume_17)
processed_sample_17 = nlp_process(cleaned_sample_17)
X_sample_17 = vectorizer.transform([processed_sample_17]).toarray()
pred_prob_17 = model.predict(X_sample_17)
predicted_label_17 = np.argmax(pred_prob_17, axis=1)[0]
predicted_category_17 = le.inverse_transform([predicted_label_17])[0]
confidence_17 = pred_prob_17.max()
print("\n--- External Validation Sample 17 (Mechanical Engineer) ---")
print(f"Predicted Category: {predicted_category_17}")
print(f"Confidence: {confidence_17:.4f}")

# Sample 18: Network Security Engineer
sample_resume_18 = """
Network Security Engineer with 5 years experience. Skills: Firewalls, Intrusion Detection, VPN, CISSP Certified. 
Education: B.Tech in Cybersecurity. Experience: Secured networks, monitored threats, implemented security policies.
"""
cleaned_sample_18 = clean_resume(sample_resume_18)
processed_sample_18 = nlp_process(cleaned_sample_18)
X_sample_18 = vectorizer.transform([processed_sample_18]).toarray()
pred_prob_18 = model.predict(X_sample_18)
predicted_label_18 = np.argmax(pred_prob_18, axis=1)[0]
predicted_category_18 = le.inverse_transform([predicted_label_18])[0]
confidence_18 = pred_prob_18.max()
print("\n--- External Validation Sample 18 (Network Security Engineer) ---")
print(f"Predicted Category: {predicted_category_18}")
print(f"Confidence: {confidence_18:.4f}")

# Sample 19: Operations Manager
sample_resume_19 = """
Operations Manager with 7 years experience. Skills: Supply Chain Management, Process Improvement, Team Leadership, Budgeting. 
Education: MBA in Operations. Experience: Optimized operations, managed teams, reduced costs, ensured quality control.
"""
cleaned_sample_19 = clean_resume(sample_resume_19)
processed_sample_19 = nlp_process(cleaned_sample_19)
X_sample_19 = vectorizer.transform([processed_sample_19]).toarray()
pred_prob_19 = model.predict(X_sample_19)
predicted_label_19 = np.argmax(pred_prob_19, axis=1)[0]
predicted_category_19 = le.inverse_transform([predicted_label_19])[0]
confidence_19 = pred_prob_19.max()
print("\n--- External Validation Sample 19 (Operations Manager) ---")
print(f"Predicted Category: {predicted_category_19}")
print(f"Confidence: {confidence_19:.4f}")

# Sample 20: PMO
sample_resume_20 = """
PMO Analyst with 5 years experience. Skills: Project Portfolio Management, Risk Assessment, MS Project, Governance. 
Education: PMP Certified. Experience: Coordinated projects, tracked progress, reported to stakeholders, mitigated risks.
"""
cleaned_sample_20 = clean_resume(sample_resume_20)
processed_sample_20 = nlp_process(cleaned_sample_20)
X_sample_20 = vectorizer.transform([processed_sample_20]).toarray()
pred_prob_20 = model.predict(X_sample_20)
predicted_label_20 = np.argmax(pred_prob_20, axis=1)[0]
predicted_category_20 = le.inverse_transform([predicted_label_20])[0]
confidence_20 = pred_prob_20.max()
print("\n--- External Validation Sample 20 (PMO) ---")
print(f"Predicted Category: {predicted_category_20}")
print(f"Confidence: {confidence_20:.4f}")